# C4 · psffit — notebook de análisis (`debug`)

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C4_codex_psf_fitting.md`](../../../docs/spec_C4_codex_psf_fitting.md)

Rehace C4 **dentro del notebook**. Es el **método canónico** de la cadena, y el único que no mide un residuo: en cada canal ajusta **dos PSF a la vez** —la primaria y el compañero— resolviendo un sistema lineal de dos amplitudes. El halo no se resta antes, se ajusta *junto con* la fuente.

Eso trae su propio modo de fallo, y es el que hay que vigilar aquí: si las dos PSF se parecen demasiado en la región de ajuste, el sistema no puede repartir la luz entre ellas. La correlación **ρ(a,b)** mide justo eso, y es el chequeo `v4_rho_ab_ok` del QC.

> **El ajuste es por canal y cuesta ~11 min para los 3681.** Por eso el notebook trae una perilla de submuestreo: cada canal se ajusta de forma independiente, así que quedarse con 1 de cada N no cambia el resultado de esos canales — verificado comparando dos submuestreos distintos, que salen **bit a bit iguales** en los canales comunes. Ponla a 1 para recorrer los 3681.

> Frente al producto **guardado** por la cadena el acuerdo es de redondeo (~1e-12 en relativo), no bit a bit como en C2 y C3: aquí hay un sistema lineal por canal, no una suma. Por eso la comparación usa `rtol=1e-9`.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

# Resolución de las figuras EN PANTALLA. `savefig` guarda a 300 dpi, pero
# lo que se ve dentro del notebook lo fija el backend inline, que va a 100
# dpi por defecto y sale borroso. `retina` dobla los píxeles sin cambiar el
# tamaño aparente; fuera de IPython no hace nada y queda el rcParam.
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Los dos radios son **la** decisión de esta etapa: definen la región donde se ajustan las dos PSF. Agrandar el del compañero mete más halo en el ajuste; encogerlo deja menos píxeles para separar las dos fuentes. Las dos cosas se ven en ρ(a,b).


In [ ]:
from musepipe.stages.stage_x03_psffit import stage_x03_config_from_run

# `project_root=ROOT`: musepipe resuelve rutas contra el cwd, que en un
# notebook es su propia carpeta, no la raíz del repo.
X03 = stage_x03_config_from_run(RUN_ID, project_root=ROOT)
STAR_RADIUS_PX    = float(X03.get('x03_star_radius_px', 20.0))
COMP_RADIUS_PX    = float(X03.get('x03_comp_radius_px', 12.0))
ERROR_MODE        = X03.get('x03_error_mode', 'auto')
N_CONTROLS        = int(X03.get('x03_control_apertures', 8))
EXCLUDE_ANGLE_DEG = float(X03.get('x03_control_exclude_angle_deg', 25.0))
BAD_WINDOWS_A     = X03.get('x03_bad_windows_A', [])
SKYLINE_WINDOWS_A = X03.get('x03_skyline_windows_A', [])
INTERPOLATED_WIN_A = X03.get('x03_interpolated_windows_A', [])
# Solo para el contraste de la seccion 8: psffit no usa apcorr (da el
# flujo total directamente), pero la apertura con la que se compara si.
APCORR_MODE       = X03.get('x03_aperture_correction', 'auto')

# Submuestreo: 1 de cada N canales. El ajuste es independiente por canal,
# así que estos salen idénticos a los de la cadena completa. Pon 1 (y ~11
# min de paciencia) para comparar los 3681.
PASO_CANALES = 20

# ---- a partir de aquí, cambia lo que quieras probar ----

print(f'radios: primaria {STAR_RADIUS_PX:.0f} px · compañero {COMP_RADIUS_PX:.0f} px'
      f' | controles {N_CONTROLS} | 1 de cada {PASO_CANALES} canales')


## 2 · Entradas

C4 extrae del **cubo de B2**, sin sustracción previa: la primaria entra en el ajuste como una de las dos componentes.


In [ ]:
qc_b3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
COMP_YX = tuple(float(v) for v in qc_b3['companion']['pos_yx'])
STAR_YX = tuple(float(v) for v in qc_b3['primary']['pos_yx'])
PSF_MODEL = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))

CUBE_PATH = SD / 'stage02_xcorr_cube_stack.fits'
with fits.open(CUBE_PATH) as h:
    CUBE_FULL = np.asarray(h['CUBES'].data, dtype=float)
    WAVE_FULL = np.asarray(h['WAVELENGTH'].data, dtype=float)
    STAT_FULL = np.asarray(h['STAT'].data, dtype=float) if 'STAT' in h else None
    _stack_bunit = str(h[0].header.get('BUNIT', '')
                       or h['CUBES'].header.get('BUNIT', '')) or None
# La unidad, con la regla de la cadena: el stack de B2 no la declara y
# `resolve_bunit` cae al cubo de entrada del run.
from musepipe.io import resolve_bunit
BUNIT = resolve_bunit(X03, stack_bunit=_stack_bunit)
UNIDAD = BUNIT or 'sin unidad declarada'
if CUBE_FULL.ndim == 4:
    CUBE_FULL = CUBE_FULL[0]
if STAT_FULL is not None and STAT_FULL.ndim == 4:
    STAT_FULL = STAT_FULL[0]

qc00 = json.loads((SD / 'stage00q_qc.json').read_text(encoding='utf-8'))
qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
m5 = qc00.get('m5_stat', {})
STAT_FACTOR = float(X03.get('x03_stat_factor_spaxel',
                            m5.get('factor_spaxel_median', 1.0)) or 1.0)
COV_FACTOR  = float(X03.get('x03_covariance_factor_box3',
                            qc01.get('stat', {}).get('covariance_factor_box3', 1.0)) or 1.0)
STAT_STATUS = str(X03.get('x03_stat_status', m5.get('status', 'unknown')))

CANALES = np.arange(0, WAVE_FULL.size, PASO_CANALES)
CUBE = CUBE_FULL[CANALES]
WAVE = WAVE_FULL[CANALES]
STAT = None if STAT_FULL is None else STAT_FULL[CANALES]
print('cubo     :', CUBE_FULL.shape, '-> se ajustan', WAVE.size, 'canales')
print('primaria :', [round(v, 2) for v in STAR_YX],
      ' compañero:', [round(v, 2) for v in COMP_YX])
sep = float(np.hypot(COMP_YX[0] - STAR_YX[0], COMP_YX[1] - STAR_YX[1]))
print(f'separación: {sep:.1f} px | radios de ajuste {STAR_RADIUS_PX:.0f}/{COMP_RADIUS_PX:.0f} px'
      f" -> las regiones {'SE SOLAPAN' if sep < STAR_RADIUS_PX + COMP_RADIUS_PX else 'no se solapan'}")
print(f'STAT     : factor={STAT_FACTOR:.3f} covarianza={COV_FACTOR:.3f} estado={STAT_STATUS}')
print('BUNIT    :', UNIDAD)


## 3 · Las funciones copiadas de `musepipe`

Incluye la dataclass del resultado, porque el ajuste la construye.

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `_npix_eff` — de `musepipe/extraction/aperture.py`
- `aperture_spectrum` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`
- `covariance_factor_for_npix` — de `musepipe/extraction/optimal.py`
- `estimate_variance_cube` — de `musepipe/extraction/optimal.py`
- `PsfFitCubeResult` — de `musepipe/extraction/psffit.py`
- `fit_region_mask` — de `musepipe/extraction/psffit.py`
- `psf_pair_design` — de `musepipe/extraction/psffit.py`
- `_correlation` — de `musepipe/extraction/psffit.py`
- `_fit_one_channel` — de `musepipe/extraction/psffit.py`
- `fit_psffit_cube` — de `musepipe/extraction/psffit.py`
- `control_psffit_spectra` — de `musepipe/extraction/psffit.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from dataclasses import dataclass
from musepipe.parallel import run_channel_chunks
from musepipe.psf import evaluate_psf_model
from typing import Sequence
import math
import numpy as np
import warnings
from dataclasses import dataclass
# `evaluate_psf_model` (C1) y `run_channel_chunks` (paralelismo) se importan
# arriba: no son lo que se ajusta aquí.

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def _npix_eff(cube_zyx: np.ndarray, weights: np.ndarray) -> np.ndarray:
    valid = np.isfinite(cube_zyx) & (weights[None, :, :] > 0)
    sumw = np.sum(weights[None, :, :] * valid, axis=(1, 2))
    sumw2 = np.sum((weights[None, :, :] ** 2) * valid, axis=(1, 2))
    out = np.full(cube_zyx.shape[0], np.nan, dtype=np.float64)
    good = sumw2 > 0
    out[good] = (sumw[good] ** 2) / sumw2[good]
    return out


def aperture_spectrum(cube_zyx, center_yx, aperture: dict) -> tuple[np.ndarray, np.ndarray]:
    """Return weighted-sum spectrum and per-channel effective pixel count."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    weighted = cube * weights[None, :, :]
    with np.errstate(invalid="ignore"):
        flux = np.nansum(weighted, axis=(1, 2)).astype(np.float64)
    npix_eff = _npix_eff(cube, weights)
    flux[~np.isfinite(npix_eff)] = np.nan
    return flux, npix_eff


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model."""

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    return (1.0 / fractions).astype(np.float64), "psf_growth_curve", norm_radius


def covariance_factor_for_npix(npix_eff, covariance_factor_box3=1.0):
    """Linearly interpolate covariance inflation between one pixel and box3."""

    vals = np.asarray(npix_eff, dtype=np.float64)
    box3 = float(covariance_factor_box3)
    if not np.isfinite(box3) or box3 <= 0:
        box3 = 1.0
    t = np.clip((vals - 1.0) / 8.0, 0.0, 1.0)
    return 1.0 + t * (box3 - 1.0)


def estimate_variance_cube(cube_zyx):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    out = np.empty_like(cube, dtype=np.float64)
    for i in range(cube.shape[0]):
        sigma = robust_sigma(cube[i])
        if not np.isfinite(sigma) or sigma <= 0:
            sigma = 1.0
        out[i] = sigma**2
    return out


@dataclass(frozen=True)
class PsfFitCubeResult:
    coeffs: np.ndarray
    covariance: np.ndarray
    chi2r: np.ndarray
    condition_number: np.ndarray
    rho_ab: np.ndarray
    rho_bc: np.ndarray
    npix: np.ndarray
    npix_eff_comp: np.ndarray
    residual_cube: np.ndarray
    model_cube: np.ndarray
    fit_mask: np.ndarray


def fit_region_mask(shape, star_yx, comp_yx, *, star_radius_px=20.0, comp_radius_px=12.0):
    ny, nx = map(int, shape)
    yy, xx = np.indices((ny, nx), dtype=np.float64)
    sy, sx = map(float, star_yx)
    cy, cx = map(float, comp_yx)
    return ((yy - sy) ** 2 + (xx - sx) ** 2 <= float(star_radius_px) ** 2) | (
        (yy - cy) ** 2 + (xx - cx) ** 2 <= float(comp_radius_px) ** 2
    )


def psf_pair_design(shape, wave_A, star_yx, comp_yx, psf_model):
    yy, xx = np.indices(shape, dtype=np.float64)
    p_star = evaluate_psf_model(
        psf_model,
        float(wave_A),
        yy - float(star_yx[0]),
        xx - float(star_yx[1]),
    )
    p_comp = evaluate_psf_model(
        psf_model,
        float(wave_A),
        yy - float(comp_yx[0]),
        xx - float(comp_yx[1]),
    )
    y0 = 0.5 * (float(star_yx[0]) + float(comp_yx[0]))
    x0 = 0.5 * (float(star_yx[1]) + float(comp_yx[1]))
    scale = max(float(np.hypot(float(comp_yx[0]) - float(star_yx[0]), float(comp_yx[1]) - float(star_yx[1]))), 1.0)
    y_scaled = (yy - y0) / scale
    x_scaled = (xx - x0) / scale
    return np.stack([p_star, p_comp, np.ones(shape), y_scaled, x_scaled], axis=-1)


def _correlation(cov, i, j):
    denom = float(cov[i, i] * cov[j, j])
    if not np.isfinite(denom) or denom <= 0:
        return np.nan
    return float(cov[i, j] / np.sqrt(denom))


def _fit_one_channel(data_2d, variance_2d, design_3d, fit_mask):
    data = np.asarray(data_2d, dtype=np.float64)
    variance = np.asarray(variance_2d, dtype=np.float64)
    design = np.asarray(design_3d, dtype=np.float64)
    valid = np.asarray(fit_mask, dtype=bool) & np.isfinite(data) & np.isfinite(variance) & (variance > 0)
    valid &= np.all(np.isfinite(design), axis=-1)
    n = int(np.count_nonzero(valid))
    p = design.shape[-1]
    if n <= p:
        coeff = np.full(p, np.nan, dtype=np.float64)
        cov = np.full((p, p), np.nan, dtype=np.float64)
        return coeff, cov, np.nan, np.inf, np.nan, np.nan, n, np.nan, np.full_like(data, np.nan), np.full_like(data, np.nan)

    a = design[valid].reshape(n, p)
    y = data[valid]
    var = variance[valid]
    sw = 1.0 / np.sqrt(var)
    aw = a * sw[:, None]
    yw = y * sw
    coeff, *_ = np.linalg.lstsq(aw, yw, rcond=None)
    normal = aw.T @ aw
    cov = np.linalg.pinv(normal)
    model = np.tensordot(design, coeff, axes=([-1], [0]))
    resid = data - model
    dof = max(1, n - p)
    chi2r = float(np.nansum((resid[valid] ** 2) / var) / dof)

    col_norm = np.linalg.norm(aw, axis=0)
    safe = col_norm > 0
    if np.all(safe):
        condition = float(np.linalg.cond(aw / col_norm[None, :]))
    else:
        condition = np.inf
    rho_ab = _correlation(cov, 0, 1)
    rho_bc_vals = [_correlation(cov, 1, j) for j in (2, 3, 4)]
    rho_bc = float(np.nanmax(np.abs(rho_bc_vals))) if np.any(np.isfinite(rho_bc_vals)) else np.nan
    pcomp = a[:, 1]
    npix_eff = np.nan
    if np.sum(pcomp**2) > 0:
        npix_eff = float((np.sum(pcomp) ** 2) / np.sum(pcomp**2))
    return coeff, cov, chi2r, condition, rho_ab, rho_bc, n, npix_eff, model, resid


def fit_psffit_cube(
    cube_zyx,
    variance_zyx,
    wave_A,
    star_yx,
    comp_yx,
    psf_model,
    *,
    star_radius_px=20.0,
    comp_radius_px=12.0,
    n_jobs=1,
) -> PsfFitCubeResult:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    variance = np.asarray(variance_zyx, dtype=np.float64)
    wave = np.asarray(wave_A, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    if variance.shape != cube.shape:
        raise ValueError("variance_zyx shape must match cube_zyx.")
    if wave.ndim != 1 or wave.size != cube.shape[0]:
        raise ValueError("wave_A must match cube spectral length.")

    nz, ny, nx = cube.shape
    fit_mask = fit_region_mask((ny, nx), star_yx, comp_yx, star_radius_px=star_radius_px, comp_radius_px=comp_radius_px)
    coeffs = np.full((nz, 5), np.nan, dtype=np.float64)
    cov = np.full((nz, 5, 5), np.nan, dtype=np.float64)
    chi2r = np.full(nz, np.nan, dtype=np.float64)
    cond = np.full(nz, np.nan, dtype=np.float64)
    rho_ab = np.full(nz, np.nan, dtype=np.float64)
    rho_bc = np.full(nz, np.nan, dtype=np.float64)
    npix = np.zeros(nz, dtype=np.int32)
    npix_eff = np.full(nz, np.nan, dtype=np.float64)
    model_cube = np.full_like(cube, np.nan, dtype=np.float64)
    residual_cube = np.full_like(cube, np.nan, dtype=np.float64)

    def _fit_range(z0, z1):
        # Per-channel work identical to the serial loop; disjoint output slots.
        for z in range(z0, z1):
            design = psf_pair_design((ny, nx), wave[z], star_yx, comp_yx, psf_model)
            row = _fit_one_channel(cube[z], variance[z], design, fit_mask)
            coeffs[z], cov[z], chi2r[z], cond[z], rho_ab[z], rho_bc[z], npix[z], npix_eff[z], model_cube[z], residual_cube[z] = row

    run_channel_chunks(_fit_range, nz, n_jobs=n_jobs)
    return PsfFitCubeResult(
        coeffs=coeffs,
        covariance=cov,
        chi2r=chi2r,
        condition_number=cond,
        rho_ab=rho_ab,
        rho_bc=rho_bc,
        npix=npix,
        npix_eff_comp=npix_eff,
        residual_cube=residual_cube,
        model_cube=model_cube,
        fit_mask=fit_mask,
    )


def control_psffit_spectra(
    cube_zyx,
    variance_zyx,
    wave_A,
    star_yx,
    comp_yx,
    psf_model,
    *,
    star_radius_px=20.0,
    comp_radius_px=12.0,
    n_controls=8,
    exclude_angle_deg=25.0,
    n_jobs=1,
) -> tuple[list[tuple[int, int]], np.ndarray, np.ndarray]:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    _, ny, nx = cube.shape
    controls = same_radius_control_positions(
        comp_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(np.ceil(comp_radius_px)) + 1,
    )
    star_specs = []
    comp_specs = []
    for center in controls:
        fit = fit_psffit_cube(
            cube,
            variance_zyx,
            wave_A,
            star_yx,
            center,
            psf_model,
            star_radius_px=star_radius_px,
            comp_radius_px=comp_radius_px,
            n_jobs=n_jobs,
        )
        star_specs.append(fit.coeffs[:, 0])
        comp_specs.append(fit.coeffs[:, 1])
    if not controls:
        return controls, np.empty((0, cube.shape[0])), np.empty((0, cube.shape[0]))
    return controls, np.asarray(star_specs, dtype=np.float64), np.asarray(comp_specs, dtype=np.float64)


## 4 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "d8e1fd8bb88d",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:_npix_eff": "32ecf15dcce9",
    "musepipe/extraction/aperture.py:aperture_spectrum": "214c68e30b47",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "8ed04307abec",
    "musepipe/extraction/optimal.py:covariance_factor_for_npix": "bfff5c5c63d8",
    "musepipe/extraction/optimal.py:estimate_variance_cube": "52dd9aee1ded",
    "musepipe/extraction/psffit.py:PsfFitCubeResult": "149f25f484cb",
    "musepipe/extraction/psffit.py:fit_region_mask": "90cdb2410161",
    "musepipe/extraction/psffit.py:psf_pair_design": "c61762fea651",
    "musepipe/extraction/psffit.py:_correlation": "4e2f284568ab",
    "musepipe/extraction/psffit.py:_fit_one_channel": "5fe11d3ec8fb",
    "musepipe/extraction/psffit.py:fit_psffit_cube": "706a881da6bc",
    "musepipe/extraction/psffit.py:control_psffit_spectra": "713d8f221856"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name),
                    None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} C4')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · La región de ajuste y las dos PSF

La máscara es la unión de dos discos. Y las dos columnas de la matriz de diseño son las dos PSF normalizadas: **si se parecen dentro de la máscara, el ajuste no puede separarlas**, y eso es exactamente lo que mide ρ(a,b) más abajo.


In [ ]:
from matplotlib.colors import LogNorm

mask = fit_region_mask(CUBE.shape[1:], STAR_YX, COMP_YX,
                       star_radius_px=STAR_RADIUS_PX, comp_radius_px=COMP_RADIUS_PX)
iz = int(np.argmin(np.abs(WAVE - 7500)))
design = psf_pair_design(CUBE.shape[1:], WAVE[iz], STAR_YX, COMP_YX, PSF_MODEL)
print('píxeles en la región de ajuste:', int(mask.sum()))

ys, xs = np.nonzero(mask)
sl = (slice(max(ys.min() - 3, 0), ys.max() + 4),
      slice(max(xs.min() - 3, 0), xs.max() + 4))
sy0, sx0 = sl[0].start, sl[1].start
# `psf_pair_design` apila las columnas en el ÚLTIMO eje: la forma es
# (ny, nx, 5), no (5, ny, nx). Con `design[i]` se cogía la fila i de la
# imagen, no la componente i, y los paneles salían en blanco.
print('forma de la matriz de diseño:', design.shape,
      '-> columnas: PSF primaria, PSF compañero, constante, y, x')
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].imshow(mask[sl], origin='lower', cmap='gray')
axes[0].set_title('región de ajuste (unión de dos discos)', fontsize=9)
# Las dos PSF comparten escala LOG y los MISMOS límites: la pregunta es
# si se parecen dentro de la máscara, y con escalas distintas dos
# perfiles diferentes se ven idénticos.
compos = [design[..., 0][sl], design[..., 1][sl]]
juntos = np.concatenate([c[np.isfinite(c) & (c > 0)].ravel() for c in compos])
norma = LogNorm(vmin=max(float(np.nanpercentile(juntos, 55)), 1e-12),
                vmax=float(np.nanmax(juntos)))
for ax, img, t in ((axes[1], compos[0], 'PSF de la primaria'),
                   (axes[2], compos[1], 'PSF del compañero')):
    im = ax.imshow(img, origin='lower', cmap='magma', norm=norma)
    ax.contour(mask[sl], levels=[0.5], colors='tab:cyan', linewidths=0.8)
    ax.set_title(f'{t}  ·  LOG', fontsize=9)
    cb = fig.colorbar(im, ax=ax, fraction=0.046)
    cb.set_label('PSF normalizada [adim.]', fontsize=7)
for ax in axes:
    ax.plot(STAR_YX[1] - sx0, STAR_YX[0] - sy0, '*', color='w', ms=11, mec='k')
    ax.plot(COMP_YX[1] - sx0, COMP_YX[0] - sy0, '+', color='tab:cyan', ms=9)
    ax.set_xlabel('x [px]')
axes[0].set_ylabel('y [px]')
fig.suptitle(f'λ = {WAVE[iz]:.0f} Å · contorno cian = borde de la región de ajuste',
             fontsize=9)
fig.tight_layout(); plt.show()

# El número que resume el panel: cuánto se parecen DENTRO de la máscara.
a = design[..., 0][mask]; b = design[..., 1][mask]
fin = np.isfinite(a) & np.isfinite(b)
coseno = float(np.dot(a[fin], b[fin])
               / np.sqrt(np.dot(a[fin], a[fin]) * np.dot(b[fin], b[fin])))
print(f'solape de las dos columnas en la máscara: {coseno:.4f}'
      '   (1 = indistinguibles, 0 = ortogonales)')


## 6 · El ajuste, canal a canal

Dos amplitudes por canal, con sus covarianzas. Los tres diagnósticos que importan:

- **χ²ᵣ ~ 1** dice que el modelo describe el dato con el error que declara el STAT (verificación V1 de la spec).
- **ρ(a,b)** es la degeneración: con \|ρ\|→1 el ajuste no puede decidir cuánta luz es de cada fuente, y el error real del compañero es mucho mayor que el formal. A esta separación se espera \|ρ\| < 0.3 (`v4_rho_ab_ok`).
- El **número de condición** avisa de lo mismo por la vía numérica.


In [ ]:
variance = (estimate_variance_cube(CUBE) if STAT is None
            else np.asarray(STAT, dtype=float) * STAT_FACTOR)
res = fit_psffit_cube(CUBE, variance, WAVE, STAR_YX, COMP_YX, PSF_MODEL,
                      star_radius_px=STAR_RADIUS_PX, comp_radius_px=COMP_RADIUS_PX,
                      n_jobs=1)
print(f'χ²ᵣ mediano   = {float(np.nanmedian(res.chi2r)):.3f}')
print(f'|ρ(a,b)| mediano = {float(np.nanmedian(np.abs(res.rho_ab))):.3f}'
      f'  (p95 {float(np.nanpercentile(np.abs(res.rho_ab), 95)):.3f})')
print(f'condición mediana = {float(np.nanmedian(res.condition_number)):.1f}')

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
a1.plot(WAVE, res.chi2r, lw=0.7); a1.axhline(1.0, color='tab:red', ls='--', lw=0.8)
a1.set_ylabel('χ²ᵣ'); a1.set_ylim(0, np.nanpercentile(res.chi2r, 99))
a2.plot(WAVE, res.rho_ab, lw=0.7, color='tab:purple')
for lim in (-0.3, 0.3):
    a2.axhline(lim, color='tab:red', ls='--', lw=0.8)
a2.set_ylabel('ρ(a,b)'); a2.set_xlabel('λ [Å]')
a1.set_title('¿describe el modelo al dato? ¿y puede separar las dos fuentes?', fontsize=9)
fig.tight_layout(); plt.show()


## 7 · Errores y controles

El error formal sale de la covarianza del ajuste, inflado por el factor de covarianza espacial. El empírico se mide re-ajustando **el mismo par de PSF** en posiciones de control al mismo radio: mide la estabilidad del ajuste, no el ruido de fotones — y por eso en D2 viaja en columna aparte.


In [ ]:
cov = covariance_factor_for_npix(res.npix_eff_comp, COV_FACTOR)
comp_var = res.covariance[:, 1, 1] * cov
star_var = res.covariance[:, 0, 0] * cov
controls_yx, star_ctrl, comp_ctrl = control_psffit_spectra(
    CUBE, variance, WAVE, STAR_YX, COMP_YX, PSF_MODEL,
    star_radius_px=STAR_RADIUS_PX, comp_radius_px=COMP_RADIUS_PX,
    n_controls=N_CONTROLS, exclude_angle_deg=EXCLUDE_ANGLE_DEG, n_jobs=1)
comp_err_emp = (robust_sigma_axis0(comp_ctrl) if comp_ctrl.shape[0] >= 2
                else np.full(WAVE.size, robust_sigma(res.coeffs[:, 1])))
star_err_emp = (robust_sigma_axis0(star_ctrl) if star_ctrl.shape[0] >= 2
                else np.full(WAVE.size, robust_sigma(res.coeffs[:, 0])))
usable = (STAT is not None and str(ERROR_MODE).lower() != 'empirical'
          and STAT_STATUS.lower() != 'red')
comp_err = np.sqrt(np.clip(comp_var, 0.0, np.inf)) if usable else comp_err_emp
star_err = np.sqrt(np.clip(star_var, 0.0, np.inf)) if usable else star_err_emp
modo = 'stat' if usable else 'empirical'
print(f'{len(controls_yx)} controles | modo de error: {modo}')
print(f'  compañero: formal {float(np.nanmedian(np.sqrt(comp_var))):8.2f}'
      f'  empírico {float(np.nanmedian(comp_err_emp)):8.2f}')
print(f'  primaria : formal {float(np.nanmedian(np.sqrt(star_var))):8.2f}'
      f'  empírico {float(np.nanmedian(star_err_emp)):8.2f}')


## 8 · Los dos espectros

C4 entrega **dos** productos: el compañero (`spec_psffit_object.fits`, el canónico de toda la cadena) y la primaria (`spec_psffit_star.fits`, que D2 calibra desde 2026-07-25). Aquí `apcorr` es 1: el ajuste devuelve directamente el flujo total de cada fuente, no el de una apertura.

### La primaria, contrastada con fotometría de apertura

El flujo de la primaria es la única de las dos componentes que se puede **verificar por otro camino**: es brillante y está aislada, así que una **apertura circular simple** —la misma `aperture_spectrum` que usa C2, con la corrección de apertura de la PSF de C1— debe dar lo mismo. Si las dos curvas se separan, el problema no está en el compañero: está en el modelo de PSF o en la curva de crecimiento, y entonces el flujo del compañero (que **no** se puede contrastar así) hereda ese error.

Por eso van juntas, con la diferencia debajo. Es el mismo espíritu que el chequeo `v3_star_scale` de la spec, pero con la cuenta a la vista y con el radio de la apertura como perilla: si la corrección de apertura fuera correcta, **el resultado no debería depender del radio**.

### Sobre el suavizado

Sí, las curvas llevan **mediana móvil** (`SUAVIZADO_CH` canales) — antes iba fija a 11 y no se decía, que es justo lo que hace que un espectro parezca mejor de lo que es. Ahora el dato **por canal** va detrás en gris y el suavizado es una perilla: ponla a 1 y se ve el espectro crudo.\n\n> Y hay un **segundo** motivo por el que esto se ve más liso que el mismo espectro en C2: el notebook ajusta **1 de cada `PASO_CANALES` canales** (20 por defecto). La línea «por canal» son ~175 puntos, no 3681. El submuestreo no cambia el valor de ningún canal —el ajuste es independiente canal a canal, y por eso la comparación con la cadena sale idéntica— pero sí cambia el aspecto. Pon `PASO_CANALES = 1` y unos 11 minutos si quieres verlo entero.


In [ ]:
from musepipe.spectral import median_filter_1d

SUAVIZADO_CH = 11        # 1 = sin suavizar, y se ve el dato crudo
RADIO_APERTURA_PX = 10.0 # apertura circular sobre la primaria

comp_flux = res.coeffs[:, 1]
star_flux = res.coeffs[:, 0]
suave = lambda v: (median_filter_1d(v, SUAVIZADO_CH) if SUAVIZADO_CH > 1
                   else np.asarray(v, dtype=float))

# --- el otro camino: fotometría de apertura sobre la primaria ---
# Misma función que usa C2 para el compañero, y la misma corrección de
# apertura de la PSF de C1, para que las dos curvas signifiquen lo mismo
# (flujo TOTAL de la fuente) y sean comparables sin más.
APERTURA_STAR = {'kind': 'circle', 'radius_px': float(RADIO_APERTURA_PX),
                 'name': f'star_r{RADIO_APERTURA_PX:g}'}
star_ap_raw, star_npix = aperture_spectrum(CUBE, STAR_YX, APERTURA_STAR)
star_apcorr, _modo_ap, _nr = aperture_correction_from_psf(
    WAVE, APERTURA_STAR, PSF_MODEL, center_yx=STAR_YX, correction_mode=APCORR_MODE)
star_ap = star_ap_raw * star_apcorr
print(f'apertura sobre la primaria: círculo r={RADIO_APERTURA_PX:g} px'
      f' ({float(np.nanmedian(star_npix)):.0f} px), apcorr mediano'
      f' {float(np.nanmedian(star_apcorr)):.2f}× -> recoge el'
      f' {100 / float(np.nanmedian(star_apcorr)):.0f}% de la PSF')

# Solo donde el ajuste corrió (el submuestreo deja canales sin ajustar).
ajustados = np.isfinite(star_flux)
dif = np.where(ajustados, star_ap - star_flux, np.nan)
rel = 100.0 * dif / np.where(np.abs(star_flux) > 0, star_flux, np.nan)
print(f'primaria: psffit vs apertura -> diferencia relativa mediana'
      f' {float(np.nanmedian(rel)):+.1f}%'
      f' (p16..p84: {float(np.nanpercentile(rel[np.isfinite(rel)], 16)):+.1f}'
      f' .. {float(np.nanpercentile(rel[np.isfinite(rel)], 84)):+.1f}%)')

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.0), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(WAVE, star_flux, lw=0.3, color='0.8')
a1.plot(WAVE, star_ap, lw=0.3, color='0.8')
a1.plot(WAVE, suave(star_flux), lw=1.2, color='k', label='psffit (ajuste de dos PSF)')
a1.plot(WAVE, suave(star_ap), lw=1.2, color='tab:orange',
        label=f'apertura r={RADIO_APERTURA_PX:g} px × apcorr')
a1.set_ylabel(f'primaria [{UNIDAD}]')
a1.set_title(f'la primaria por dos caminos independientes'
             f'  (líneas: mediana móvil de {SUAVIZADO_CH} ch; gris: por canal)',
             fontsize=9)
a1.legend(fontsize=8)
a2.axhline(0, color='0.6', lw=0.7)
a2.plot(WAVE, dif, lw=0.3, color='0.8')
a2.plot(WAVE, suave(dif), lw=1.1, color='tab:purple')
fin_d = np.isfinite(dif)
if fin_d.any():
    a2.set_ylim(*np.nanpercentile(dif[fin_d], [1, 99]))
a2.set_ylabel(f'apertura − psffit\n[{UNIDAD}]', fontsize=8)
a2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.fill_between(WAVE, -comp_err_emp, comp_err_emp, color='0.85',
                label='±σ empírico (controles)')
ax.plot(WAVE, comp_flux, lw=0.3, color='0.6', label='por canal (sin suavizar)')
if SUAVIZADO_CH > 1:
    ax.plot(WAVE, suave(comp_flux), lw=1.2, color='tab:blue',
            label=f'mediana móvil {SUAVIZADO_CH} ch')
ax.axvline(6562.8, color='tab:red', ls=':', label='Hα')
fin_c = np.isfinite(comp_flux)
if fin_c.any():
    ax.set_ylim(*np.nanpercentile(comp_flux[fin_c], [1, 99]))
ax.set_ylabel(f'compañero [{UNIDAD}]'); ax.set_xlabel('λ [Å]')
ax.set_title('el compañero: el dato por canal, y su mediana móvil encima', fontsize=9)
ax.legend(fontsize=8); fig.tight_layout(); plt.show()
print(f'razón primaria/compañero (mediana): '
      f'{float(np.nanmedian(star_flux) / np.nanmedian(comp_flux)):.0f}×')


## 9 · Comparación con la cadena

Los dos productos, **solo en los canales ajustados** (el submuestreo no cambia el resultado de un canal: el ajuste es independiente canal a canal). Con las perillas por defecto debe salir idéntico.


In [ ]:
from musepipe.extraction.product import SpectrumProduct

def compara(nombre, mio_flux, mio_err, fichero, rtol=1e-9):
    ref = SpectrumProduct.read(SD / fichero)
    ok = True
    print(f'{nombre} vs {fichero}:')
    for clave, a, b in (('flujo', mio_flux, np.asarray(ref.flux, float)[CANALES]),
                        ('error', mio_err, np.asarray(ref.flux_err, float)[CANALES])):
        fin = np.isfinite(a) & np.isfinite(b)
        d = np.abs(a - b)[fin]
        ig = np.isclose(a[fin], b[fin], rtol=rtol, atol=0.0)
        print(f'   {clave:6s} idénticos {100 * ig.mean():6.2f}% de {fin.sum()} canales'
              f' | máx |Δ| = {d.max():.3e}')
        ok &= bool(ig.all())
    return ok

ok = compara('compañero', comp_flux, comp_err, 'spec_psffit_object.fits')
ok &= compara('primaria ', star_flux, star_err, 'spec_psffit_star.fits')
print()
print('IDÉNTICO: la copia reproduce la cadena.' if ok else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, revisa el chequeo de deriva.')
